# 🌌 HollyShit Phase 0 v3: 去空腔各向同性五师流形蒸馏 (`SensoryFuser + GodEncoder`)

- **五师魔搭按需流式串流 (`catclever/emb_npy`)**: 联合融合 `bge-m3` (1024d)、`gte_Qwen2-7B` (3584d)、`xiaobu-v2` (1792d)、`Youtu` (2048d)、`Conan-v1` (1792d) 共 18 个 50w 宏分块（`8,656,199` 条子句），启用 **`embedding_only=True` 纯向量流式模式，零本地/云盘 Parquet 依赖，零 CPU 分词损耗**。
- **去空腔各向同性神之流形**: 五路逆向重构头 (`MultiViewReconstructionHeads`) + 共识关系相似度矩阵蒸馏 (`Rel`) + Dirichlet 单纯形视角云与近邻测地 Mixup 填腔 (`Cloud`) + 跨批次队列全谱 VICReg 协方差白化与超球均匀性 (`Iso + Unif`) + 连续子句轨迹动量与非对称下一句检索 (`Momentum + Next`)。
- **本地 NVMe SSD 缓存与单块控流**: 分块缓存在 `/content/embs_cache`，配合 `--no-prefetch` 单块内存驻留（峰值内存约 `~10.2 GB`，每消费完一块自动删除 `.npz` 释放磁盘）与 `--keep_last_ckpts 3` 自动滚动清理旧权重。
- **推荐 Colab 运行时规格**: **L4 / A100 GPU**（默认大内存），或 **T4 GPU + 务必开启「高内存 (High-RAM >= 25GB)」**。

In [ ]:
# 0. 清理旧版 Phase 0 v3 Checkpoint (如需从零全新训练可取消注释并运行本单元格；默认保留以支持 --auto_resume 自动断点续训)
# !rm -rf /content/drive/MyDrive/hollyshit/checkpoints/p0_v3/p0_v3_5models_*


In [ ]:
import os
import shutil
from google.colab import drive

# 1. 挂载 Google Drive
drive.mount('/content/drive')

# 2. 安装流式下载与权重序列化依赖 (纯 PyTorch 原生算子，秒级就绪)
!pip install -q modelscope safetensors pyarrow

# 3. 可选读取 Colab Secrets 中的 MODELSCOPE_API_TOKEN (公开库 catclever/emb_npy 匿名亦可直连)
try:
    from google.colab import userdata
    ms_tok = userdata.get('MODELSCOPE_API_TOKEN')
    if ms_tok:
        os.environ['MODELSCOPE_API_TOKEN'] = ms_tok.strip()
        print('🔑 已从 Colab Secrets 加载 MODELSCOPE_API_TOKEN')
except Exception:
    pass

# 4. 硬件规格与高内存 (High-RAM) 安全自检
import torch

def _get_system_ram_gb() -> float:
    if os.path.exists('/proc/meminfo'):
        with open('/proc/meminfo', 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith('MemTotal:'):
                    kb = int(line.split()[1])
                    return kb / (1024.0 ** 2)
    return 0.0

ram_gb = _get_system_ram_gb()
disk_free_gb = shutil.disk_usage('/content').free / (1024.0 ** 3)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (未检测到 GPU!)'
vram_gb = (torch.cuda.get_device_properties(0).total_memory / (1024.0 ** 3)) if torch.cuda.is_available() else 0.0

print('\n' + '=' * 62)
print(f'🔥 GPU 设备   : {gpu_name} ({vram_gb:.1f} GB VRAM)')
print(f'🧠 系统内存   : {ram_gb:.1f} GB RAM')
print(f'💾 本地 SSD   : /content 可用空间 {disk_free_gb:.1f} GB')
print('=' * 62)
if ram_gb > 0 and ram_gb < 18.0:
    print('⚠️ [重要警告] 当前系统内存 < 18 GB！单块 50w 五师特征解压约占 10.2 GB RAM。')
    print('   请在菜单栏「代码执行程序 -> 更改运行时类型」中开启【高内存 (High-RAM)】后再启动训练！')


In [ ]:
import os

# 从 GitHub 拉取纯代码并初始化 training/core 子模块 (如已存在则 git pull)
if not os.path.exists('/content/HollyShit'):
    !git clone https://github.com/catclever/HollyShit.git /content/HollyShit
    %cd /content/HollyShit
    !git submodule update --init training/core
else:
    %cd /content/HollyShit
    !git pull
    !git submodule update --init training/core

# 确保云盘 Checkpoint 与数据目录存在，并建立项目软链接
os.makedirs('/content/drive/MyDrive/hollyshit/checkpoints/p0_v3', exist_ok=True)
os.makedirs('/content/drive/MyDrive/hollyshit/data', exist_ok=True)
!ln -sfn /content/drive/MyDrive/hollyshit/checkpoints checkpoints
!ln -sfn /content/drive/MyDrive/hollyshit/data data
print('✅ 仓库与云盘软链接就绪:', os.getcwd())


In [ ]:
import json
import os
import re
import torch
from modelscope.hub.api import HubApi

# 1. 创建本地高速 NVMe SSD 分块流式缓存目录 (严禁放在 Google Drive!)
local_embs_cache = '/content/embs_cache'
os.makedirs(local_embs_cache, exist_ok=True)

# 2. 探测魔搭 catclever/emb_npy 五师分块矩阵连通性与完整性
with open('dataset_config.json', 'r', encoding='utf-8') as f:
    ds_cfg = json.load(f)

ms_repo_id = ds_cfg['modelscope_repo_id']
models = ds_cfg['base_models']
patterns = ds_cfg['chunk_name_patterns']
emb_dims = ds_cfg['emb_dims']

api = HubApi()
if os.environ.get('MODELSCOPE_API_TOKEN'):
    try:
        api.login(os.environ['MODELSCOPE_API_TOKEN'])
    except Exception as e:
        print(f'⚠️ Token 鉴权跳过 (使用匿名访问): {e}')

remote_files = [item.get('Path', item.get('Name', '')) for item in api.get_dataset_files(ms_repo_id, recursive=True)]
per_model = {}
for m_name, dim in zip(models, emb_dims):
    prefix = patterns[m_name].split('/')[0]
    bounds = {}
    for rf in remote_files:
        rf_norm = rf.replace('\\', '/')
        if rf_norm.endswith('.npz') and (rf_norm.startswith(prefix + '/') or f'/{prefix}/' in rf_norm):
            mm = re.search(r'chunk_(\d+)_(\d+)\.npz', os.path.basename(rf_norm))
            if mm:
                bounds[int(mm.group(1))] = int(mm.group(2))
    per_model[m_name] = bounds
    print(f'  📡 {m_name:<9} (dim={dim:<4}): 探测到 {len(bounds)} 个远端 .npz 分块')

common_starts = sorted(set.intersection(*[set(b.keys()) for b in per_model.values()]))
assert len(common_starts) > 0, '未探测到共享分块，请检查网络或 dataset_config.json！'
total_clauses = min(per_model[m][common_starts[-1]] for m in models)
print(f'✅ 魔搭流式源 [{ms_repo_id}] 握手成功！共 {len(common_starts)} 个对齐宏分块，总计 {total_clauses:,} 条子句。')

# 3. 按 GPU 档位 (L4/A100 vs T4) 自适应设定最优超参并注入环境变量
vram_gb = (torch.cuda.get_device_properties(0).total_memory / (1024.0 ** 3)) if torch.cuda.is_available() else 0.0
ram_gb = 0.0
if os.path.exists('/proc/meminfo'):
    with open('/proc/meminfo', 'r', encoding='utf-8') as f:
        for line in f:
            if line.startswith('MemTotal:'):
                ram_gb = int(line.split()[1]) / (1024.0 ** 2)
                break

if vram_gb >= 22.0 and ram_gb >= 24.0:
    # L4 (24GB VRAM / ~53GB RAM) 或 A100 (40/80GB VRAM / ~83GB RAM)
    p0_cfg = {
        'P0_BATCH_SIZE': '512',
        'P0_TRIPLET_BATCH': '128',
        'P0_LR': '4e-4',
        'P0_WARMUP': '1000',
        'P0_TOTAL_STEPS': '50750',
    }
    tier_label = 'L4 / A100 高吞吐大批量档 (Batch=512)'
else:
    # T4 (16GB VRAM + High-RAM)
    p0_cfg = {
        'P0_BATCH_SIZE': '256',
        'P0_TRIPLET_BATCH': '64',
        'P0_LR': '3e-4',
        'P0_WARMUP': '2000',
        'P0_TOTAL_STEPS': '101500',
    }
    tier_label = 'T4 稳健标准档 (Batch=256)'

os.environ.update(p0_cfg)
print(f'⚙️ 已激活训练配置 [{tier_label}]: {p0_cfg}')


In [ ]:
# ==============================================================================
# 🚀 启动 Phase 0 v3: 去空腔各向同性五师神之流形训练
# - 纯向量流式模式 (embedding_only=True): 无需本地/云盘 Parquet 语料
# - --no-prefetch: 严格单块驻留 /content/embs_cache，消费完自动删除旧 .npz 分块
# - --auto_resume & --keep_last_ckpts 3: 自动断点续训，滚动保留最近 3 个检查点省云盘空间
# ==============================================================================
!python -u training/train_phase0_v3_cuda.py \
  --config_file dataset_config.json \
  --data_dir /content/embs_cache \
  --download_cache_dir /content/embs_cache \
  --no-prefetch \
  --batch_size $P0_BATCH_SIZE \
  --triplet_batch_size $P0_TRIPLET_BATCH \
  --epochs 3 \
  --lr $P0_LR \
  --warmup_steps $P0_WARMUP \
  --total_steps $P0_TOTAL_STEPS \
  --out_dir /content/drive/MyDrive/hollyshit/checkpoints/p0_v3 \
  --ckpt_prefix p0_v3_5models \
  --eval_steps 200 \
  --save_steps 5000 \
  --keep_last_ckpts 3 \
  --auto_resume


In [ ]:
import os
import shutil
import time
import torch
from google.colab import drive, runtime
from training.train_phase0_v3_cuda import resolve_latest_phase0_v3_checkpoint

drive_ckpt_root = '/content/drive/MyDrive/hollyshit/checkpoints/p0_v3'
ckpt_prefix = 'p0_v3_5models'
latest_ckpt = resolve_latest_phase0_v3_checkpoint(drive_ckpt_root, ckpt_prefix)

print('=' * 68)
if latest_ckpt:
    print(f'🏆 最新有效 Phase 0 v3 检查点: {latest_ckpt}')
    ms_path = os.path.join(latest_ckpt, 'manifold_stats.pt')
    if os.path.isfile(ms_path):
        ms = torch.load(ms_path, map_location='cpu')
        hm = ms.get('health_metrics', {})
        print(f"   • global_step : {ms.get('global_step')}")
        print(f"   • arch        : emb_dims={ms.get('emb_dims')} -> d_model={ms.get('d_model')} -> z_dim={ms.get('z_dim')} (sphere={ms.get('sphere_scale')})")
        if hm:
            print(
                f"   • health      : PR={hm.get('participation_ratio', 0.0):.1f} | "
                f"ReconCos={hm.get('mean_teacher_recon_cos', 0.0):.4f} | "
                f"RelSpearman={hm.get('relational_spearman', 0.0):.4f} | "
                f"MidpointCos={hm.get('midpoint_cavity_recon_cos', 0.0):.4f} | "
                f"CloudTop1={hm.get('cloud_view_top1_acc', 0.0):.4f} | "
                f"NextTop1={hm.get('next_clause_top1_acc', 0.0):.4f} | "
                f"AdjAUC={hm.get('adjacent_auc', 0.0):.4f}"
            )

    # 校验下游 Phase 1 v2 蒸馏所需的双格式教师权重并同步固定别名目录 best
    required_exports = [
        'sense_fuser.safetensors',
        'god_encoder.safetensors',
        'sense_fuser.pt',
        'god_encoder.pt',
        'manifold_stats.pt',
        'training_args.json',
    ]
    missing = [fn for fn in required_exports[:2] if not os.path.isfile(os.path.join(latest_ckpt, fn))]
    assert not missing, f'缺少下游 Phase 1 必需权重文件: {missing}'

    best_alias_dir = os.path.join(drive_ckpt_root, f'{ckpt_prefix}_best')
    os.makedirs(best_alias_dir, exist_ok=True)
    for fn in required_exports:
        src_file = os.path.join(latest_ckpt, fn)
        if os.path.isfile(src_file):
            shutil.copy2(src_file, os.path.join(best_alias_dir, fn))
    print(f'🔗 已同步下游 Phase 1 v2 固定教师目录 (--p0_ckpt): {best_alias_dir}')
else:
    print('⚠️ 未在云盘目录中检测到已保存的有效检查点。')
print('=' * 68)

# 强制刷写 Google Drive FUSE 写入缓存，防止断连丢权重
print('🔄 正在强制刷写 Google Drive 写入缓存...')
try:
    drive.flush_and_unmount()
    print('✅ Google Drive 缓存已完全落盘！')
except Exception as e:
    print(f'⚠️ 缓存刷写提示: {e}')

# 可选：训练完成后自动释放 GPU 运行时 (节省 Colab 计算单元)
# time.sleep(5)
# runtime.unassign()
